# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library and references the Croissant schema and all data elements exclusively by their `@id` fields.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The metadata object can be accessed from the dataset, and all schema entities are referenced by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
List the available record sets and their `@id` values. Then, for each record set, show the available fields and their IDs. We use only `@id` for reference.


In [ ]:
# Retrieve and list all record sets by their @id
record_sets = list(dataset.record_sets.keys())

print("Available Record Sets (@id):")
for record_set_id in record_sets:
    rs = dataset.record_sets[record_set_id]
    print(f"- {record_set_id} | name: {getattr(rs, 'name', None)}")
    # List the available fields for this record set
    if hasattr(rs, 'fields'):
        print(f"  Fields in {record_set_id} (@id):")
        for field_id in rs.fields:
            field = dataset.fields[field_id]
            print(f"    - {field_id} | name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load the data from each record set, referencing them by their `@id` fields. Each record set is loaded into a pandas DataFrame, and columns are referenced by their `@id` as well.

In [ ]:
# Define which record sets to extract (use all found above)
all_record_set_ids = list(dataset.record_sets.keys())
dataframes = {}

for record_set_id in all_record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Loaded {len(dataframes[record_set_id])} records, columns (by @id): {list(dataframes[record_set_id].columns)}\n")
    else:
        print("  No records found.\n")

# For demonstration, pick the *first* available record set for analysis
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Selected main record set for further analysis: {main_rs_id}")
    print("Sample data:")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply example data manipulations: filter by a numeric field's `@id`, normalize its values, and group by a key categorical field (by `@id`).

Please check the available numeric and categorical field IDs above, and substitute accordingly if necessary. The notebook uses `@id` strings to reference each field.

In [ ]:
# Identify a numeric field and a group field by @id
# For demonstration, we use common clinical dataset fields; adjust as appropriate for your dataset.

main_df = dataframes[main_rs_id]

# Attempt to infer numeric and categorical columns
numeric_field_id = None
group_field_id = None

# Try to pick 'Age' or similar; otherwise default to the first float/int field
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: pick first float/int column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

# Try to pick 'Sex', 'Gender', or similar for grouping
for col in main_df.columns:
    if any(s in col.lower() for s in ['sex', 'gender', 'msi']):
        group_field_id = col
        break

if numeric_field_id:
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 0
    print(f"Filtering records with {numeric_field_id} > {threshold:.2f}")
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"{len(filtered_df)} records remain.")
    print(filtered_df[[numeric_field_id]].head())
    # Normalize the numeric field
    col_norm = numeric_field_id + '_normalized'
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} ({col_norm}):")
    print(filtered_df[[numeric_field_id, col_norm]].head())
    
    # Group by a categorical field
    if group_field_id and group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df)
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field (referenced by its `@id`) and relationships to the group field. All visualizations use `@id` labels for clarity.


In [ ]:
# Visualization by @id
if numeric_field_id:
    plt.figure(figsize=(8,4))
    main_df[numeric_field_id].hist(bins=15)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id and group_field_id in main_df:
        plt.figure(figsize=(8,4))
        main_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant metadata and tabular records from the FAIR^2 dataset using the `mlcroissant` library.
- Enumerate available record sets, fields, and columns by their `@id`.
- Extract tabular data and explore it using pandas, referencing all columns and filters by their schema IDs.
- Conduct initial exploration and visualize data distributions within the cohort.

Continue your project by applying further statistical analysis or machine learning to these DataFrames using the column `@id`s for maximum schema portability.